<a href="https://colab.research.google.com/github/umesh23042003/Statistical-Learning-e22222/blob/main/assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""Refactored_DataInspector.ipynb"""

import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
from IPython.display import HTML, display
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from scipy.stats import chi2_contingency

class DataInspector:
    """
    A robust toolkit designed for data cleaning, exploration,
    and interactive visualization within Google Colab environments.
    """
    def __init__(self):
        self.df = None
        self.numeric_cols = []
        self.categorical_cols = []

    # === 1. DATA INGESTION & SANITIZATION ===
    def upload_data(self):
        """Handles local file uploads interactively in Google Colab."""
        print("Please upload your CSV file:")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return

        file_name = list(uploaded.keys())[0]
        garbage_strings = ['?', '$n/a$', 'n/a', 'N/A', 'NULL', 'null', ' ']
        self.df = pd.read_csv(io.BytesIO(uploaded[file_name]), na_values=garbage_strings)
        print(f"\nSuccessfully loaded {file_name}!")
        self._auto_type_correction()

    def _auto_type_correction(self):
        """Force-converts columns to numeric types if it doesn't result in all nulls."""
        if self.df is None: return

        for col in self.df.columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if not converted.isna().all():
                self.df[col] = converted

        self.numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

    # === 2. STRUCTURAL ANALYSIS & CLEANING ===
    def get_summary(self):
        """Displays dimension counts, type breakdowns, and a preview of the dataset."""
        if self.df is None:
            print("No dataset loaded.")
            return

        print(f"Dataset Shape: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
        print(f"Numerical Columns ({len(self.numeric_cols)}): {self.numeric_cols}")
        print(f"Categorical Columns ({len(self.categorical_cols)}): {self.categorical_cols}\n")
        print("--- First 20 Rows Preview ---")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='median', fill_value=None):
        """Imputes missing values using mean, median, mode, or a constant strategy."""
        if self.df is None: return

        for col in self.df.columns:
            if self.df[col].isna().sum() == 0: continue

            if col in self.numeric_cols:
                strat_map = {
                    'mean': self.df[col].mean(),
                    'median': self.df[col].median(),
                    'mode': self.df[col].mode()[0] if not self.df[col].mode().empty else 0
                }
                val = fill_value if strategy == 'constant' else strat_map.get(strategy, self.df[col].median())
                self.df[col].fillna(val, inplace=True)
            else:
                fallback = self.df[col].mode()
                val = fill_value if strategy == 'constant' else (fallback[0] if not fallback.empty else "Unknown")
                self.df[col].fillna(val, inplace=True)

        print(f"Missing values imputed using '{strategy}' strategy.")

    def remove_duplicates(self):
        """Prunes exact duplicate rows from the active DataFrame."""
        if self.df is None: return
        initial_rows = len(self.df)
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_rows - len(self.df)} exact duplicate rows.")

    def handle_outliers(self, columns=None, find_and_delete=True):
        """IQR-based outlier detection system to flag or automatically delete rows."""
        if self.df is None: return
        cols_to_check = columns or self.numeric_cols
        rows_to_drop = set()

        for col in cols_to_check:
            Q1, Q3 = self.df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

            outliers = self.df[(self.df[col] < lower_bound) | (self.df[col] > upper_bound)].index

            if find_and_delete:
                rows_to_drop.update(outliers)
            else:
                print(f"Column '{col}' has {len(outliers)} outliers outside range ({lower_bound:.2f}, {upper_bound:.2f})")

        if find_and_delete and rows_to_drop:
            self.df.drop(index=list(rows_to_drop), inplace=True)
            print(f"Dropped {len(rows_to_drop)} rows containing outliers.")

    def delete_columns(self):
        """Interactive column deletion accepting comma-separated user inputs."""
        cols_input = input("Enter column names to delete (comma-separated): ")
        to_delete = [c.strip() for c in cols_input.split(',') if c.strip() in self.df.columns]
        self.df.drop(columns=to_delete, inplace=True)
        self._auto_type_correction()
        print(f"Deleted columns: {to_delete}")

    def delete_rows(self):
        """Interactive row indices deletion via comma-separated user inputs."""
        rows_input = input("Enter row indices to delete (comma-separated): ")
        try:
            to_delete = [int(r.strip()) for r in rows_input.split(',') if r.strip().isdigit()]
            self.df.drop(index=to_delete, inplace=True)
            print(f"Deleted row indices: {to_delete}")
        except Exception as e:
            print(f"Error deleting rows: {e}")

    # === 3. FEATURE ENGINEERING PREPARATION ===
    def extract_normalized_numeric_data(self, method='standard'):
        """Scales numeric data using 'minmax', 'standard' (Z-score), or 'robust' methods."""
        if self.df is None or not self.numeric_cols: return pd.DataFrame()

        scalers = {'minmax': MinMaxScaler(), 'robust': RobustScaler(), 'standard': StandardScaler()}
        scaler = scalers.get(method, StandardScaler())
        scaled_data = scaler.fit_transform(self.df[self.numeric_cols])
        return pd.DataFrame(scaled_data, columns=self.numeric_cols, index=self.df.index)

    def extract_normalized_categorical_data(self, method='onehot'):
        """Encodes categorical strings using 'onehot', 'ordinal', or 'uniform'."""
        if self.df is None or not self.categorical_cols: return pd.DataFrame()

        if method == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, drop='first')
            encoded = encoder.fit_transform(self.df[self.categorical_cols].astype(str))
            return pd.DataFrame(encoded, columns=encoder.get_feature_names_out(self.categorical_cols), index=self.df.index)

        encoder = OrdinalEncoder()
        df_encoded = pd.DataFrame(
            encoder.fit_transform(self.df[self.categorical_cols].astype(str)),
            columns=self.categorical_cols,
            index=self.df.index
        )
        if method == 'uniform':
            df_encoded = (df_encoded - df_encoded.min()) / (df_encoded.max() - df_encoded.min() + 1e-9)
        return df_encoded

    def create_normalized_data_df(self, numeric_method='standard', categorical_method='onehot'):
        """Creates a single merged clean DataFrame ready for Machine Learning training."""
        return pd.concat([
            self.extract_normalized_numeric_data(method=numeric_method),
            self.extract_normalized_categorical_data(method=categorical_method)
        ], axis=1)

    # === 4. ADVANCED INTERACTIVE VISUALIZATION ===
    def plot_numerical(self, column_names):
        """Generates a 3-panel subplot layout for provided numeric columns."""
        for col in [c for c in column_names if c in self.numeric_cols]:
            fig = make_subplots(rows=1, cols=3, subplot_titles=('Violin Plot', 'Index Scatter', 'Histogram'))
            fig.add_trace(go.Violin(x=self.df[col], box_visible=True, points='all', name=col), row=1, col=1)
            fig.add_trace(go.Scatter(y=self.df[col], mode='markers', marker=dict(opacity=0.6)), row=1, col=2)
            fig.add_trace(go.Histogram(x=self.df[col]), row=1, col=3)
            fig.update_layout(title_text=f"Univariate Profiling: {col}", showlegend=False, height=400)
            fig.show()

    def plot_relationship(self, var1, var2):
        """Detects column variable metadata types to auto-select smart plots."""
        if var1 not in self.df.columns or var2 not in self.df.columns: return

        v1_num, v2_num = (var1 in self.numeric_cols), (var2 in self.numeric_cols)

        if v1_num and v2_num:
            fig = px.scatter(self.df, x=var1, y=var2, trendline="ols", title=f"Scatter Trend: {var1} vs {var2}")
        elif not v1_num and not v2_num:
            fig = px.density_heatmap(self.df, x=var1, y=var2, text_auto=True, title=f"Grouped Categorical: {var1} vs {var2}")
        else:
            cat_var, num_var = (var1, var2) if not v1_num else (var2, var1)
            fig = px.box(self.df, x=cat_var, y=num_var, points="all", title=f"Distribution: {num_var} by {cat_var}")
        fig.show()

    # === 5. DEEP STATISTICAL INSIGHTS ===
    def _calculate_cramers_v(self, c1, c2):
        """Helper to calculate Cramér's V for two categorical columns."""
        conf_matrix = pd.crosstab(self.df[c1], self.df[c2])
        if conf_matrix.size == 0: return 0.0
        chi2 = chi2_contingency(conf_matrix)[0]
        n_obs = conf_matrix.sum().sum()
        r, k = conf_matrix.shape
        phi2corr = max(0, (chi2 / n_obs) - ((k-1)*(r-1))/(n_obs-1))
        rcorr = r - ((r-1)**2)/(n_obs-1)
        kcorr = k - ((k-1)**2)/(n_obs-1)
        denom = min((kcorr-1), (rcorr-1))
        return np.sqrt(phi2corr / denom) if denom > 0 else 0.0

    def plot_all_associations_heatmap(self):
        """Calculates a global association matrix matching all unique data-type relationships."""
        if self.df is None: return
        cols = self.df.columns
        assoc_matrix = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)

        for i, c1 in enumerate(cols):
            for c2 in cols[i+1:]:
                if c1 in self.numeric_cols and c2 in self.numeric_cols:
                    val = self.df[c1].corr(self.df[c2], method='pearson')
                elif c1 in self.categorical_cols and c2 in self.categorical_cols:
                    val = self._calculate_cramers_v(c1, c2)
                else:
                    num_col, cat_col = (c1, c2) if c1 in self.numeric_cols else (c2, c1)
                    try:
                        val = self.df[num_col].corr(self.df[cat_col].astype('category').cat.codes)
                    except:
                        val = 0.0

                assoc_matrix.loc[c1, c2] = assoc_matrix.loc[c2, c1] = val

        fig = px.imshow(assoc_matrix, text_auto=".2f", color_continuous_scale='RdBu_r',
                        title="Unified Association Heatmap (Pearson / Cramér's V / Mixed Proxy)")
        fig.show()

    def display_image(self, result_dict):
        """Helper to safely display raw Plotly HTML snippets wrapped by PlottingMethods."""
        if isinstance(result_dict, dict) and "html" in result_dict:
            display(HTML(result_dict["html"]))
        else:
            print("Invalid format for rendering.")

class PlottingMethods:
    """ Handles granular custom chart generation returning embedded figures. """

    @staticmethod
    def _generate_html(fig):
        """Centralized try/except & rendering wrapper."""
        try:
            return {"status": "success", "html": fig.to_html(include_plotlyjs='cdn', full_html=False)}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def plot_bar_chart(self, x, y, data, color=None, barmode='group', title=None):
        """Creates a custom Plotly bar chart return statement dictionary."""
        return self._generate_html(px.bar(data, x=x, y=y, color=color, barmode=barmode, title=title, text_auto=True))

    def plot_pie_chart(self, names, values, data, hole=0.4, title=None):
        """Generates a responsive customized donut slice chart layout."""
        return self._generate_html(px.pie(data, names=names, values=values, hole=hole, title=title))

    def plot_histogram(self, x, data, bins=None, title=None):
        """Plots standard or explicitly binned interval histograms."""
        try:
            fig = px.histogram(data, x=x, title=title)
            if bins is not None:
                fig.update_traces(xbins=dict(start=min(bins), end=max(bins), size=(max(bins)-min(bins))/len(bins)))
            return {"status": "success", "html": fig.to_html(include_plotlyjs='cdn', full_html=False)}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def display_image(self, result):
        """Renders custom PlottingMethods dictionary outputs inside Colab notebooks."""
        if result.get("status") == "success":
            display(HTML(result["html"]))
        else:
            print(f"Plotting Error: {result.get('message')}")

# Instantiate classes
inspector = DataInspector()
plotter = PlottingMethods()

# 1. Ingest a dataset directly using a URL for testing
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(titanic_url)
inspector._auto_type_correction() # Run the type separation engine

# 2. Review structure summaries
inspector.get_summary()

# 3. Clean and impute missing values
inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()

# 4. Perform an IQR-based outlier pass on numeric values
inspector.handle_outliers(columns=['Age', 'Fare'], find_and_delete=False)

# 5. Extract a fully prepared DataFrame matching modern ML layouts
ml_ready_df = inspector.create_normalized_data_df(numeric_method='robust', categorical_method='onehot')
print("\n--- Processed Feature Matrix Head ---")
display(ml_ready_df.head(5))

# 6. Deep Interactive Exploratory Graphics
inspector.plot_numerical(['Age', 'Fare'])
inspector.plot_relationship('Pclass', 'Fare')
inspector.plot_all_associations_heatmap()

# 7. Use the explicit modular PlottingMethods class
res = plotter.plot_pie_chart(names='Sex', values='PassengerId', data=inspector.df, title='Gender Breakdown')
plotter.display_image(res)

Dataset Shape: 891 rows, 12 columns
Numerical Columns (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
Categorical Columns (4): ['Name', 'Sex', 'Cabin', 'Embarked']

--- First 20 Rows Preview ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


Missing values imputed using 'median' strategy.
Removed 0 exact duplicate rows.
Column 'Age' has 66 outliers outside range (2.50, 54.50)
Column 'Fare' has 116 outliers outside range (-26.72, 65.63)


/tmp/ipykernel_598/4147883779.py:79: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[col].fillna(val, inplace=True)
/tmp/ipykernel_598/4147883779.py:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.


--- Processed Feature Matrix Head ---


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,-1.000000,0.0,0.0,-0.461538,1.0,0.0,0.000000,-0.312011,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.997753,1.0,-2.0,0.769231,1.0,0.0,0.000000,2.461242,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.995506,1.0,0.0,-0.153846,0.0,0.0,0.000000,-0.282777,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.993258,1.0,-2.0,0.538462,1.0,0.0,-0.396270,1.673732,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.991011,0.0,0.0,0.538462,0.0,0.0,0.444557,-0.277363,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
